# 01 · Explore Olist
Learn the data before the LLM does. Every trap found here is written into `data/schema_docs.yaml`, which is what the model sees.

Run from the repo root with the database loaded (`python -m db.load_data`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # notebook lives in notebooks/
import pandas as pd
from sqlalchemy import create_engine
from askdb.config import settings

engine = create_engine(settings.database_url)  # read-only askdb_reader
q = lambda sql: pd.read_sql(sql, engine)

## Table sizes

In [ ]:
tables = ['orders','order_items','order_payments','order_reviews','customers','products','category_translation','sellers','geolocation']
pd.DataFrame({'table': tables, 'rows': [q(f'SELECT COUNT(*) n FROM {t}').n[0] for t in tables]})

## Trap 1 · Order status
97% of orders are `delivered`. Revenue questions should only count those.

In [ ]:
q('SELECT order_status, COUNT(*) AS orders FROM orders GROUP BY 1 ORDER BY 2 DESC')

## Trap 2 · `customer_id` is per order
There are 99,441 `customer_id` values but only 96,096 real people (`customer_unique_id`). Counting `customer_id` overcounts customers and finds **zero** repeat customers.

In [ ]:
q('''SELECT COUNT(DISTINCT customer_id) AS customer_ids,
          COUNT(DISTINCT customer_unique_id) AS people FROM customers''')

In [ ]:
q('''SELECT COUNT(*) FILTER (WHERE n_by_id > 1) AS repeat_by_customer_id,
          (SELECT COUNT(*) FROM (SELECT c.customer_unique_id FROM orders o JOIN customers c USING (customer_id)
                                 GROUP BY 1 HAVING COUNT(*) > 1) s) AS repeat_by_unique_id
   FROM (SELECT customer_id, COUNT(*) n_by_id FROM orders GROUP BY 1) s''')

## Trap 3 · Several payment rows per order
Summing `payment_value` without grouping by order, or counting rows instead of orders, gives wrong answers.

In [ ]:
q('''SELECT n_payments, COUNT(*) AS orders FROM
   (SELECT order_id, COUNT(*) n_payments FROM order_payments GROUP BY 1) s
   GROUP BY 1 ORDER BY 1''')

## Trap 4 · Categories are Portuguese
Join `category_translation`. 610 products have no category and 2 categories have no translation.

In [ ]:
q('''SELECT p.product_category_name, t.product_category_name_english, COUNT(*) AS products
   FROM products p LEFT JOIN category_translation t USING (product_category_name)
   WHERE t.product_category_name IS NULL GROUP BY 1, 2''')

## Trap 5 · Reviews
`review_id` is not unique and some orders have several reviews.

In [ ]:
q('''SELECT (SELECT COUNT(*) FROM (SELECT review_id FROM order_reviews GROUP BY 1 HAVING COUNT(*) > 1) a) AS dup_review_ids,
          (SELECT COUNT(*) FROM (SELECT order_id FROM order_reviews GROUP BY 1 HAVING COUNT(*) > 1) b) AS orders_with_2plus_reviews''')

## Trap 6 · Geolocation duplicates
1M rows for 19K zip prefixes: aggregate per prefix before joining or rows multiply.

In [ ]:
q('SELECT COUNT(*) AS rows, COUNT(DISTINCT geolocation_zip_code_prefix) AS prefixes FROM geolocation')

## Time range
Sep 2016 – Oct 2018, but 2016 and Sep–Oct 2018 are nearly empty: be careful with growth numbers at the edges.

In [ ]:
q('''SELECT date_trunc('month', order_purchase_timestamp)::date AS month, COUNT(*) AS orders
   FROM orders GROUP BY 1 ORDER BY 1''').plot(x='month', y='orders', figsize=(10, 3));

## Revenue by category (the README example)

In [ ]:
q('''SELECT t.product_category_name_english AS category, SUM(oi.price) AS revenue
   FROM order_items oi JOIN orders o USING (order_id) JOIN products p USING (product_id)
   JOIN category_translation t USING (product_category_name)
   WHERE o.order_status = 'delivered' AND EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017
   GROUP BY 1 ORDER BY 2 DESC LIMIT 5''')